In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.utils.spark_session import spark
import pyspark.sql.functions


spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")


spark.conf.set("spark.sql.shuffle.partitions", "8")

26/06/30 15:40:15 WARN Utils: Your hostname, mangoman12bam-HP-Laptop-15-fc0xxx resolves to a loopback address: 127.0.1.1; using 10.0.0.123 instead (on interface wlo1)
26/06/30 15:40:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/30 15:40:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
project_root = Path.cwd().parent


df_fact = spark.read.parquet(str(project_root / "data/curated/fact_requests")).sample(fraction=0.03, seed=42)
df_host = spark.read.parquet(str(project_root / "data/curated/dim_host"))
df_time = spark.read.parquet(str(project_root / "data/curated/dim_timestamp"))
df_status = spark.read.parquet(str(project_root / "data/curated/dim_status"))
df_endpoint = spark.read.parquet(str(project_root / "data/curated/dim_endpoint"))

df_fact.createOrReplaceTempView("fact_requests")
df_host.createOrReplaceTempView("host")
df_time.createOrReplaceTempView("time")
df_status.createOrReplaceTempView("status")
df_endpoint.createOrReplaceTempView("endpoint")


In [ ]:
spark.sql("""
    SELECT
        e.endpoint,
        COUNT(*) AS total,
        SUM(CASE WHEN s.status IN ('4xx', '5xx') THEN 1 ELSE 0 END) AS errors,
        SUM(CASE WHEN s.status IN ('4xx', '5xx') THEN 1 ELSE 0 END) / COUNT(*) AS error_rate
    FROM fact_requests f
    JOIN endpoint e ON f.endpt_key = e.endpt_key
    JOIN status s ON f.status_key = s.status_key
    GROUP BY e.endpoint
    ORDER BY error_rate DESC
    LIMIT 10
""").show()
# df_fact.printSchema()


26/06/30 15:40:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 15:>                                                         (0 + 1) / 1]

In [ ]:
spark.sql("""
    SELECT /*+ BROADCAST(e) */
        e.extracted AS endpoint_category,
        PERCENTILE_APPROX(f.bytes, 0.95) AS p95_bytes
    FROM fact_requests f
    JOIN dim_endpoint e ON f.endpt_key = e.endpt_key
    GROUP BY e.extracted
""").show()

In [ ]:
spark.sql("""
    SELECT
        t.hour,
        COUNT(*) AS total_requests,
        ROUND(AVG(f.bytes), 0) AS avg_bytes,
        SUM(CASE WHEN s.status IN ('4xx', '5xx') THEN 1 ELSE 0 END) AS errors,
        ROUND(
            SUM(CASE WHEN s.status IN ('4xx', '5xx') THEN 1 ELSE 0 END) / COUNT(*),
            4
        ) AS error_rate
    FROM fact_requests f
    JOIN time t ON f.timestamp_key = t.timestamp_key
    JOIN status s ON f.status_key = s.status_key
    GROUP BY t.hour
    ORDER BY t.hour
""").show()